# Exercise 5 - Validation Set Approach for Logistic Regression

#### Objective of this exercise is to use the Default data set to estimate the test error of logistic regression models using the validation set approach

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from ISLP import load_data

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score

In [2]:
Default = load_data("Default")

# Convert categorical response to binary
Default["default"] = (Default["default"] == "Yes").astype(int)
Default["student"] = (Default["student"] == "Yes").astype(int)

In [3]:
X = Default[["income", "balance"]]
y = Default["default"]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

print("="*60)
print("Logistic Regression")
print("="*60)

print("Intercept:", model.intercept_[0])
print("Coefficients:")
coef = pd.DataFrame({"Variable": X.columns, "Coefficient": model.coef_[0]})
print(coef)

Logistic Regression
Intercept: -11.54046791864163
Coefficients:
  Variable  Coefficient
0   income     0.000021
1  balance     0.005647


In [4]:
def validation_error(seed, predictors):
    X = Default[predictors]
    y = Default["default"]
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.5, random_state=seed)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_valid)[:,1]
    pred = (prob > 0.5).astype(int)
    error = np.mean(pred != y_valid)
    return error

In [5]:
print("\n" + "="*60)
print("Validation Set Error")
print("="*60)

error = validation_error(seed=1, predictors=["income","balance"])
print("Validation Error =", error)


Validation Set Error
Validation Error = 0.025


In [6]:
print("\n" + "="*60)
print("Three Different Splits")
print("="*60)

for seed in [1,2,3]:
    err = validation_error(seed, ["income","balance"])
    print(f"Seed {seed}: {err:.4f}")


Three Different Splits
Seed 1: 0.0250
Seed 2: 0.0248
Seed 3: 0.0248


In [7]:
print("\n" + "="*60)
print("Including Student")
print("="*60)

for seed in [1,2,3]:
    err = validation_error(seed, ["income","balance","student"])
    print(f"Seed {seed}: {err:.4f}")


Including Student
Seed 1: 0.0262
Seed 2: 0.0250
Seed 3: 0.0252


- Using the validation set approach with a 50-50 train-test split, the model achieved a validation error of 2.50%. This indicates that the logistic regression model correctly classified approximately 97.5% of the observations in the validation set, demonstrating good predictive performance.
- The estimated test errors are nearly identical across the three random splits. This suggests that the logistic regression model is stable and that the estimated prediction error is not highly sensitive to the particular training and validation sets selected. Minor differences are expected due to random sampling.
- Including the student variable does not reduce the validation error. In fact, the prediction error is almost identical and is slightly larger for some random splits. Therefore, there is little evidence that the student predictor provides additional predictive information once income and balance are already included in the model.

# Exercise 6 - Estimating Logistic Regression Standard Errors using Bootstrap

#### Objective of this exercise is to compare methods for estimating the standard errors of logistic regression coefficients on the Default data set.

In [12]:
np.random.seed(1)
Default = load_data("Default")
Default["default"] = (Default["default"] == "Yes").astype(int)

X = Default[["income", "balance"]]
X = sm.add_constant(X)
y = Default["default"]

glm = sm.GLM(y, X, family=sm.families.Binomial()).fit()

print(glm.summary())
print("\nStandard Errors")
print(glm.bse)

                 Generalized Linear Model Regression Results                  
Dep. Variable:                default   No. Observations:                10000
Model:                            GLM   Df Residuals:                     9997
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -789.48
Date:                Wed, 05 Aug 2026   Deviance:                       1579.0
Time:                        22:44:03   Pearson chi2:                 6.95e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1256
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -11.5405      0.435    -26.544      0.0

In [13]:
def boot_fn(data, index):
    sample = data.iloc[index]

    X = sample[["income", "balance"]]
    X = sm.add_constant(X)
    y = sample["default"]

    model = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    return model.params

B = 1000
boot_coef = np.zeros((B,3))
n = len(Default)

for i in range(B):
    index = np.random.choice(n, n, replace=True)
    boot_coef[i,:] = boot_fn(Default,index)

boot_coef = pd.DataFrame(boot_coef, columns=["const", "income", "balance"])

print("\n" + "="*60)
print("Bootstrap Standard Errors")
print("="*60)
print(boot_coef.std())


Bootstrap Standard Errors
const      0.448999
income     0.000005
balance    0.000233
dtype: float64


In [14]:
comparison = pd.DataFrame({"GLM Std Error": glm.bse, "Bootstrap Std Error": boot_coef.std()})

print("\n" + "="*60)
print("Comparison")
print("="*60)
print(comparison)


Comparison
         GLM Std Error  Bootstrap Std Error
const         0.434772             0.448999
income        0.000005             0.000005
balance       0.000227             0.000233


The bootstrap standard errors are very similar to those obtained from the GLM output. For the intercept, the bootstrap estimate (≈0.449) is close to the GLM estimate (≈0.435). Likewise, the bootstrap standard errors for income (≈0.000005) and balance (≈0.000233) are nearly identical to the GLM estimates. This indicates that the standard errors computed using the asymptotic GLM formula are reliable for this data set, and the bootstrap provides empirical confirmation of these estimates.